In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from datetime import datetime, timedelta
import numpy as np
import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

In [ ]:
def remove_message_timestamp(message):
    regex = r"'time': \d+, "
    new_message = re.sub(regex, '', str(message))
    regex = r"'dpId': \d+, "
    new_message = re.sub(regex, '', str(new_message))
    return eval(new_message)

def remove_invalid_codes(message):
    valid_codes = ['switch_led', 'switch_1', 'presence_state', 'doorcontact_state']
    result = []
    for state in message:
        if state['code'] in valid_codes:
            result.append(state)
    if len(result) > 0:
        return result
    else:
        return np.nan

def time_interval(dt, minutes=60):
    total_seconds = timedelta(hours=dt.hour, minutes=dt.minute, seconds=dt.second).total_seconds()
    interval = total_seconds // (minutes * 60)
    return int(interval)

def device_change_heatmap_1(df):
    df['interval'] = df['timeStamp'].apply(time_interval)
    routine_dict = pd.DataFrame({'interval': range(24)})
    for device in unique_devices:
        a = df[df['devId'] == device].groupby('interval')['devId'].count().to_frame().rename({'devId': device}, axis=1)
        routine_dict = routine_dict.merge(a, on='interval', how='left')

    routine_dict = routine_dict.drop('interval', axis=1)
    routine_map = routine_dict.fillna(0).T
    fig = plt.figure(figsize=(25, 5))
    
    heatmap = sns.heatmap(routine_map, vmin=-1, vmax=24, annot=True, cmap='Blues', fmt='.0f',
                          linecolor='black', annot_kws={'size': 16, 'weight': 'heavy'})
    heatmap.collections[0].colorbar.ax.tick_params(labelsize=22)
    plt.xticks(fontsize=22)
    plt.yticks(fontsize=22, rotation=0)
    plt.xlabel('Time of day', fontsize=22)
    return fig

def device_change_heatmap_Teste(df, unique_devices):
    df_meio_semana = df.query('day_week != "Sunday" and day_week != "Saturday"')
    df_fim_semana = df.query('day_week == "Sunday" or day_week == "Saturday"')

    df_meio_semana['interval'] = df_meio_semana['timeStamp'].apply(time_interval)
    df_fim_semana['interval'] = df_fim_semana['timeStamp'].apply(time_interval)
    routine_meio_semana = pd.DataFrame({'interval': range(24)})
    routine_fim_semana = pd.DataFrame({'interval': range(24)})

    for device in unique_devices:
        a = df_meio_semana[df_meio_semana['devId'] == device].groupby('interval')['devId'].count().to_frame().rename({'devId': device}, axis=1)
        b = df_fim_semana[df_fim_semana['devId'] == device].groupby('interval')['devId'].count().to_frame().rename({'devId': device}, axis=1)

        routine_meio_semana = routine_meio_semana.merge(a, on='interval', how='left')
        routine_fim_semana = routine_fim_semana.merge(b, on='interval', how='left')

    routine_meio_semana = routine_meio_semana.drop('interval', axis=1)
    routine_meio_semana = routine_meio_semana.fillna(0).T

    routine_fim_semana = routine_fim_semana.drop('interval', axis=1)
    routine_fim_semana = routine_fim_semana.fillna(0).T


    fig, axes = plt.subplots(2, 1, figsize=(25, 5))
    heatmap = sns.heatmap(routine_meio_semana, ax=axes[0], vmin=-1, vmax=24, annot=True, cmap='Greens', fmt='.0f',
                          linecolor='black', annot_kws={'size': 16, 'weight': 'heavy'})
    heatmap.collections[0].colorbar.ax.tick_params(labelsize=22)
    axes[0].set_title('Weekdays', fontsize=16)
    axes[0].set_xlabel('Time of day', fontsize=12)

    heatmap = sns.heatmap(routine_fim_semana, ax=axes[1], vmin=-1, vmax=24, annot=True, cmap='Greens', fmt='.0f',
                          linecolor='black', annot_kws={'size': 16, 'weight': 'heavy'})
    heatmap.collections[0].colorbar.ax.tick_params(labelsize=22)
    axes[1].set_title('Weekend', fontsize=16)
    axes[1].set_xlabel('Time of day', fontsize=12)

    plt.subplots_adjust(wspace=0.7, hspace=0.7)
    return fig

In [ ]:
def trata_dados_reais(caminho:str):
    message_list = []
    hour_list = []
    with open(caminho, "r") as json:
        data = json.read()
        for line in data.split('\n')[1:-1]:
            hour_list.append(eval(line)['hora'])
            dict_msg = eval(eval(line)['mensagem'].replace('true', 'True').replace('false', 'False'))['bizData']
            message_list.append(dict_msg)
        json.close()
    
    df = pd.DataFrame(message_list).rename(columns={'properties': 'message'})
    df['timeStamp'] = hour_list
    df['message'] = df['message'].astype(str)
    
    dicionario = {
        'ebfe6c248a7dfe6910qdcb': 'Plug_fan',
        'eb061b979815289561tyqf': 'Sensor_presence',
        'eb31770a1d7812125degzr': 'Light_bulb',
        'eb176a71685a57c19arlbp': 'abertura',
        'ebcc9b86347718a3808ezt': 'Plug_pc',
        'ebf1d890916d1a73b4vtnv': 'temp_humidade'
    }
    
    df['devId'] = df['devId'].apply(lambda x: dicionario[x])
    
    df['message'] = df['message'].apply(remove_message_timestamp)
    df['message'] = df['message'].apply(remove_invalid_codes)
    df['timeStamp'] = df['timeStamp'].apply(lambda string: datetime.strptime(string, '%Y-%m-%dT%H:%M:%S.%f'))
    
    df = df[df['message'].notna()].reset_index(drop=True)
    df["day_week"] = df["timeStamp"].dt.day_name()
    
    unique_devices = ['Sensor_presence', 'Plug_fan', 'Light_bulb', 'Plug_pc']
    
    for device in unique_devices:
        result = (df[df['devId'] == device]['message'].shift(1) == df[df['devId'] == device]['message']).sum()
        print(f'{device}: {result}')
    
    remove_index = []
    for device in unique_devices:
        result = df[df['devId'] == device][(df[df['devId'] == device]['message'].shift(1) == df[df['devId'] == device]['message']).values].index.tolist()
        if len(result) > 0:
            remove_index.extend(result)
    df = df.drop(remove_index)
    return df

In [ ]:
def trata_dados_gerados(caminho:str):
    df_gerado = pd.read_csv(caminho)
    colunas = ['devId', 'productKey', 'message', 'timeStamp']
    
    dicionario = {
        'QUARTO_PLUG_02-004': 'Plug_fan',
        'QUARTO_SENSOR_PRESENCA-002': 'Sensor_presence',
        'QUARTO_LAMPADA-001': 'Light_bulb',
        'QUARTO_PLUG_01-003': 'Plug_pc'
    }
    
    df_gerado['devId'] = df_gerado['device'].apply(lambda x: dicionario[x])
    df_gerado = df_gerado[colunas]
    
    df_gerado['timeStamp'] = df_gerado['timeStamp'].apply(lambda string: datetime.strptime(string, '%Y-%m-%d %H:%M:%S.%f'))
    df_gerado["day_week"] = df_gerado["timeStamp"].dt.day_name()
    
    return df_gerado

In [ ]:
def time_interval_teste(dt, minutes=60):
    total_seconds = timedelta(hours=dt.hour, minutes=dt.minute, seconds=dt.second).total_seconds()
    interval = total_seconds // (minutes * 60)
    return int(interval)

def time_day_teste(dt):
    return pd.Timestamp(dt.year, dt.month, dt.day, dt.hour)

def monta_dataframe_por_dia(dados: pd.DataFrame):
    dados['day'] = dados['timeStamp'].apply(time_day_teste)

    routine = pd.DataFrame({'day': pd.date_range(start=dados['day'][0], end=dados.iloc[-1]['day'], freq='h')})
    for device in unique_devices:
        b = dados[dados['devId'] == device].groupby('day')['devId'].count().to_frame().rename({'devId': device}, axis=1)
        routine = routine.merge(b, on='day', how='left')

    # routine = routine.drop('interval', axis=1)
    routine = routine.fillna(0).T
    return routine

def gefico_linha(lista_df):
    grupo_processado = []
    for g in lista_df:    
        grupo = monta_dataframe_por_dia(g).T
        grupo['interval'] = grupo['day'].apply(time_interval_teste)
        grupo = grupo.astype({'Sensor_presence':'int', 'Plug_fan':'int', 'Light_bulb':'int', 'Plug_pc':'int'})
        grupo.drop(['day'], axis=1, inplace=True)
        grupo_processado.append(grupo.groupby(['interval']).mean())

    fig, axs = plt.subplots(2, 2, figsize=(12, 5), sharey=False)

    axs[0, 0].set_title('Sensor_presence', weight='bold')
    axs[0, 0].step(grupo_processado[0].index, grupo_processado[0]['Sensor_presence'], label='real', color='orange')
    # axs[0, 0].step(grupo_processado[1].index, grupo_processado[1]['Sensor_presence'], where='mid', label='generated_14_days', color='green')
    # axs[0, 0].step(grupo_processado[2].index, grupo_processado[2]['Sensor_presence'], where='mid', label='generated_30_days', color='Purple')
    axs[0, 0].step(grupo_processado[3].index, grupo_processado[3]['Sensor_presence'], where='mid', label='generated_60_days', color='blue')
    axs[0, 0].set_xlabel('hour of day')
    axs[0, 0].set_ylabel('status changing')
    
    axs[0, 1].set_title('Plug_fan', weight='bold')
    axs[0, 1].step(grupo_processado[0].index, grupo_processado[0]['Plug_fan'], color='orange')
    # axs[0, 1].step(grupo_processado[1].index, grupo_processado[1]['Plug_fan'], where='mid', color='green')
    # axs[0, 1].step(grupo_processado[2].index, grupo_processado[2]['Plug_fan'], where='mid', color='Purple')
    axs[0, 1].step(grupo_processado[3].index, grupo_processado[3]['Plug_fan'], where='mid', color='blue')
    axs[0, 1].set_xlabel('hour of day')
    axs[0, 1].set_ylabel('status changing')
    
    axs[1, 0].set_title('Light_bulb', weight='bold')
    axs[1, 0].step(grupo_processado[0].index, grupo_processado[0]['Light_bulb'], color='orange')
    # axs[1, 0].step(grupo_processado[1].index, grupo_processado[1]['Light_bulb'], where='mid', color='green')
    # axs[1, 0].step(grupo_processado[2].index, grupo_processado[2]['Light_bulb'], where='mid', color='Purple')
    axs[1, 0].step(grupo_processado[3].index, grupo_processado[3]['Light_bulb'], where='mid', color='blue')
    axs[1, 0].set_xlabel('hour of day')
    axs[1, 0].set_ylabel('status changing')
    
    axs[1, 1].set_title('Plug_pc', weight='bold')
    axs[1, 1].step(grupo_processado[0].index, grupo_processado[0]['Plug_pc'], color='orange')
    # axs[1, 1].step(grupo_processado[1].index, grupo_processado[1]['Plug_pc'], where='mid', color='green')
    # axs[0, 1].step(grupo_processado[2].index, grupo_processado[2]['Plug_pc'], where='mid', color='Purple')
    axs[1, 1].step(grupo_processado[3].index, grupo_processado[3]['Plug_pc'], where='mid', color='blue')
    axs[1, 1].set_xlabel('hour of day')
    axs[1, 1].set_ylabel('status changing')
    
    # plt.grid(axis='x', color='0.95')
    fig.legend(title='Legend:', loc='lower center', bbox_to_anchor=(0.5, -0.08), fancybox=False, shadow=False, borderpad=0.5, ncol=10)
    fig.suptitle('Device Behavior')
    plt.subplots_adjust(wspace=0.3, hspace=0.5)
    plt.show()
    return fig
        

# Graficos

In [ ]:
df_real = trata_dados_reais('dados.json')
df_gerado_14d = trata_dados_gerados('../dados/original/completo/dados-validacao-14-dias.csv')
df_gerado_30d = trata_dados_gerados('../dados/original/completo/dados-validacao-30-dias.csv')
df_gerado_60d = trata_dados_gerados('../dados/original/completo/dados-validacao-60-dias.csv')

In [ ]:
# Grafico de linha com comportamento dos dispositivos

grafico_comportamento = gefico_linha([df_real, df_gerado_14d, df_gerado_30d, df_gerado_60d])
grafico_comportamento.savefig("grafio_comportamento_real_gerado_60_dias.pdf", bbox_inches='tight')

In [ ]:
# gráfido de calor com contagem de utilização dos dispositivos
unique_devices = ['Sensor_presence', 'Plug_fan', 'Light_bulb', 'Plug_pc']

graph_real = device_change_heatmap_Teste(df_real, unique_devices)
graph_real.savefig("grafio_calor_real.pdf", bbox_inches='tight')

graph_gerado = device_change_heatmap_Teste(df_gerado_14d, unique_devices)
graph_gerado.savefig("grafio_calor_gerado.pdf", bbox_inches='tight')

# TEste

In [248]:
df_real = trata_dados_reais('dados.json')
df_real = monta_dataframe_por_dia(df_real).T
df_real['interval'] = df_real['day'].apply(time_interval_teste)
df_real['grupo'] = 'real'


df_gerado_14d = trata_dados_gerados('../dados/original/completo/dados-validacao-14-dias.csv')
df_gerado_14d = monta_dataframe_por_dia(df_gerado_14d).T
df_gerado_14d['interval'] = df_gerado_14d['day'].apply(time_interval_teste)
df_gerado_14d['grupo'] = 'gerado_14_dias'


df_gerado_60d = trata_dados_gerados('../dados/original/completo/dados-validacao-60-dias.csv')
df_gerado_60d = monta_dataframe_por_dia(df_gerado_60d).T
df_gerado_60d['interval'] = df_gerado_60d['day'].apply(time_interval_teste)
df_gerado_60d['grupo'] = 'gerado_60_dias'

Sensor_presence: 1044
Plug_fan: 0
Light_bulb: 5
Plug_pc: 0


In [249]:
df_concatenado = pd.concat([df_real, df_gerado_14d, df_gerado_60d], axis=0, ignore_index=True)

In [252]:
df_concatenado.to_excel('df_concatenado.xlsx', engine='xlsxwriter')

In [257]:
df_gerado_temp = pd.read_csv('../dados/original/completo/dados-validacao-temp.csv')
df_gerado_temp

,device,devId,productKey,message,sensorType,group,userAction,activityUserAction,timeStamp,space
0,CORREDOR_SENSOR_PRESENCA-027,dispf7c434c69ee54cf81abcfb1b8c79a9e3,0s9d8f9s0d8f0s9df8s0f8,"{'status': [{'code': 'presence_state', 'value'...",sensor,"['Gustavo', 'Amanda']",Amanda,CAFE_DA_MANHA,2024-09-03 07:00:40.535109,CORREDOR
1,QUARTO_01_SENSOR_PRESENCA-014,disp87451041c9f6549dfaaa0a95f83cd015,0s9d8f9s0d8f0s9df8s0f8,"{'status': [{'code': 'presence_state', 'value'...",sensor,"['Gustavo', 'Amanda']",Gustavo,ACORDAR_QUARTO_01,2024-09-03 07:00:45.741398,QUARTO_01
2,COZINHA_SENSOR_PRESENCA-021,disp16b045cc84bdde80b25c98cb4bb5575f,0s9d8f9s0d8f0s9df8s0f8,"{'status': [{'code': 'presence_state', 'value'...",sensor,"['Gustavo', 'Amanda']",Amanda,CAFE_DA_MANHA,2024-09-03 07:00:50.941075,COZINHA
3,COZINHA_CAFETEIRA-020,dispbd59f6f08242cd34668806095c642b89,98s7f076sf67s5adf7s5f7s,"{'status': [{'code': 'switch', 'value': 'ON'}]}",atuador,"['Gustavo', 'Amanda']",Amanda,CAFE_DA_MANHA,2024-09-03 07:00:55.008977,COZINHA
4,CORREDOR_SENSOR_PRESENCA-027,dispf7c434c69ee54cf81abcfb1b8c79a9e3,0s9d8f9s0d8f0s9df8s0f8,"{'status': [{'code': 'presence_state', 'value'...",sensor,"['Gustavo', 'Amanda']",Gustavo,CAFE_DA_MANHA,2024-09-03 07:00:56.395794,CORREDOR
...,...,...,...,...,...,...,...,...,...,...
1657,COZINHA_SENSOR_PRESENCA-021,disp16b045cc84bdde80b25c98cb4bb5575f,0s9d8f9s0d8f0s9df8s0f8,"{'status': [{'code': 'presence_state', 'value'...",sensor,"['Gustavo', 'Amanda']",Gustavo,ALMOCAR,2024-09-16 19:17:39.511677,COZINHA
1658,COZINHA_SENSOR_PRESENCA-021,disp16b045cc84bdde80b25c98cb4bb5575f,0s9d8f9s0d8f0s9df8s0f8,"{'status': [{'code': 'presence_state', 'value'...",sensor,"['Gustavo', 'Amanda']",Gustavo,ALMOCAR,2024-09-16 20:07:58.822544,COZINHA
1659,CORREDOR_SENSOR_PRESENCA-027,dispf7c434c69ee54cf81abcfb1b8c79a9e3,0s9d8f9s0d8f0s9df8s0f8,"{'status': [{'code': 'presence_state', 'value'...",sensor,"['Gustavo', 'Amanda']",Gustavo,FAZER_NADA,2024-09-16 20:07:58.839269,CORREDOR
1660,CORREDOR_SENSOR_PRESENCA-027,dispf7c434c69ee54cf81abcfb1b8c79a9e3,0s9d8f9s0d8f0s9df8s0f8,"{'status': [{'code': 'presence_state', 'value'...",sensor,"['Gustavo', 'Amanda']",Gustavo,FAZER_NADA,2024-09-16 20:08:09.501285,CORREDOR


In [ ]:

graph_gerado = device_change_heatmap_Teste(df_gerado_temp, unique_devices)